# BigQuery: Agentic Migration & Data Transfer (Managed MCP)

[![Open In Colab](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/bq_migration_mcp_demo.ipynb)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/bq_migration_mcp_demo.ipynb)

This notebook shows how an agent can translate legacy SQL and manage data transfers into BigQuery using the new **BQMS** and **DTS** Managed MCP servers — no custom scripts needed.

## Use Case
A migration partner needs to move data from Hive-on-GCS to BigQuery. An agent handles the work:
1.  **Translate SQL**: Convert Hive SQL to GoogleSQL using the BQMS MCP server.
2.  **Generate DDL**: Create matching `CREATE TABLE` statements for BigQuery.
3.  **Run Transfers**: Schedule a data transfer job using the DTS MCP server.

### Release Notes
- [Google Cloud Managed MCP (March 2026)](https://cloud.google.com/blog/products/data-analytics) — BQMS and DTS exposed as Managed MCP servers for agentic SQL translation and data transfer
- [ADK v1.28.0](https://github.com/google/adk-python/releases/tag/v1.28.0) — McpToolset with StreamableHTTP used as the agent interface for this demo

### Requirements
- BigQuery Migration Service and Data Transfer Service APIs enabled.
- `google-adk >= 1.28.0` installed.
- Gemini 3.1 Pro (Preview) access.

In [ ]:
# 1. Setup and Authentication
%pip install "google-adk>=1.28.0" google-genai google-cloud-bigquery google-cloud-storage nest-asyncio --quiet --index-url https://pypi.org/simple

try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab')
except ModuleNotFoundError:
    print('Not running in Colab — using Application Default Credentials (ADC)')

import os
import nest_asyncio
nest_asyncio.apply()

project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
location = 'us-central1'  # @param {type:"string"}
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = location
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

### 2. [MANDATORY] Project Configuration & Service Enablement

Both BQMS and DTS Managed MCP servers must be enabled, along with the source/target APIs.

In [ ]:
# Enable services
!gcloud services enable bigquery.googleapis.com bigquerymigration.googleapis.com bigquerydatatransfer.googleapis.com storage.googleapis.com --project={project_id}

# Enable the MCP services
!gcloud beta services mcp enable migration.googleapis.com --project={project_id}
!gcloud beta services mcp enable bigquerydatatransfer.googleapis.com --project={project_id}
print("Success: APIs and Managed MCP services enabled.")

### 3. [PREREQUISITES] Infrastructure Setup (Simulating Legacy Environment)

To simulate a Hive-to-BigQuery migration, we create a target dataset and a source bucket with sample 'Hive' data.

In [ ]:
from google.cloud import bigquery, storage

def setup_migration_prereqs():
    # 1. Setup BigQuery Target Dataset
    bq_client = bigquery.Client(project=project_id, location=location)
    dataset_id = f"{project_id}.legacy_migration_target"
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = location
    bq_client.create_dataset(dataset, exists_ok=True)
    
    # 2. Setup GCS Source (Simulating Hive External Table storage)
    storage_client = storage.Client(project=project_id)
    bucket_name = f"{project_id}-hive-source"
    bucket = storage_client.create_bucket(bucket_name, location=location) if not storage_client.lookup_bucket(bucket_name) else storage_client.get_bucket(bucket_name)
    
    # Upload dummy Hive data
    blob = bucket.blob("hive_table/part-0000.csv")
    blob.upload_from_string("1,John,Doe,2025-01-15\n2,Jane,Smith,2025-01-16")
    
    print(f"Prerequisites ready: Target Dataset '{dataset_id}' and Source Bucket 'gs://{bucket_name}/'")

setup_migration_prereqs()

### 4. Core Feature: SQL Translation & Transfer Management

We initialize the MCP toolsets and an agent specialized in migrations.

In [ ]:
from google.adk.tools.mcp_tool import McpToolset, StreamableHTTPConnectionParams
from google.adk import Agent, Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.adk.models import Gemini
from google import genai
from google.genai import types
import google.auth
from google.auth.transport.requests import Request

# Refresh token for MCP (must be fresh before each MCP call)
scopes = ["https://www.googleapis.com/auth/cloud-platform"]
creds, _ = google.auth.default(scopes=scopes)
creds.refresh(Request())

# 1. Initialize BQMS MCP Toolset
bqms_mcp = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url="https://migration.googleapis.com/mcp",
        headers={"Authorization": f"Bearer {creds.token}", "x-goog-user-project": project_id},
        timeout=30.0
    )
)

# 2. Initialize DTS MCP Toolset
dts_mcp = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url="https://bigquerydatatransfer.googleapis.com/mcp",
        headers={"Authorization": f"Bearer {creds.token}", "x-goog-user-project": project_id},
        timeout=30.0
    )
)

# 3. Gemini 3.1 Pro Preview is only available in the 'global' location.
#    We override the model client while keeping GOOGLE_CLOUD_LOCATION regional for MCP/BQ.
model = Gemini(
    model="gemini-3.1-pro-preview",
    client=genai.Client(vertexai=True, project=project_id, location='global')
)

migration_agent = Agent(
    model=model,
    name="MigrationAssistant",
    instruction="""
    You are a data migration expert. 
    Use BQMS to translate legacy SQL and DTS to manage data transfers into BigQuery.
    Always verify the translation success before suggesting a transfer.
    """,
    tools=[bqms_mcp, dts_mcp]
)

# 4. Initialize Runner
runner = Runner(
    agent=migration_agent,
    session_service=InMemorySessionService(),
    app_name="migration_mcp_demo",
    auto_create_session=True
)

async def run_migration_workflow():
    legacy_sql = "SELECT * FROM hive_table WHERE dt > '2025-01-01' LIMIT 10;"
    prompt = f"Translate this Hive SQL to GoogleSQL: {legacy_sql}"
    
    print(f"--- Task 1: Translating Legacy SQL ---")
    print(f"Source (Hive): {legacy_sql}\n")
    
    message = types.Content(parts=[types.Part(text=prompt)], role='user')
    async for event in runner.run_async(
        user_id="partner_user",
        session_id="march_session",
        new_message=message
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(f"Agent: {part.text}")
                if part.function_call:
                    print(f"[SYSTEM]: Calling tool '{part.function_call.name}'")

await run_migration_workflow()

### 5. Things to remember or know
- **No custom glue code**: Managed MCP servers handle the hard parts — SQL translation, DDL generation, and transfer scheduling — so there's nothing to build or maintain.
- **Multi-dialect translation**: The BQMS MCP server translates from Hive, Teradata, and other dialects to GoogleSQL.
- **End-to-end pipeline**: Agents can run the full Ingest → Translate → Transfer pipeline by combining BQMS and DTS MCP servers.
- **Runner pattern**: All March 2026 demos use the `Runner` for automatic session management and event streaming.
- **Availability**: Preview as of March 24-25, 2026.